In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
import random
import time
from glob import glob
from pathlib import Path

import numpy as np
import torch
from skl2onnx.helpers.onnx_helper import load_onnx_model

from service.fragment.net import Net
from service.stitching.generate_networks import generate_networks
from utilities.dataloader_generator import generate_dataloader
from utilities.load_dataset_chest_xray import load_dataset
from utilities.report import Report
from utilities.score_mapper import ScoreMapper


2025-11-23 14:53:44.218363978 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [3]:
netsFiles = sorted(glob("../_results_chest_xray/fragments/net*"))
nets = []
for index, netsFile in enumerate(netsFiles):
    fragmentFiles = sorted(glob(str(Path(netsFile) / "fragment*.onnx")))
    onnxFragments = []
    for fragmentFile in fragmentFiles:
        onnxFragment = load_onnx_model(fragmentFile)
        onnxFragments.append(onnxFragment)
    net1 = Net(onnxFragments, index)
    nets.append(net1)


In [4]:
random.seed(51)
np.random.seed(24)
torch.manual_seed(77)

K = 5
STITCH_BATCH_SIZE = 32
MAX_DEPTH = 16
THRESOULD = 0
TOTAL_THRESOULD = 0.5

RESULT_NAME = f"{int(time.time())}_result_BS_{STITCH_BATCH_SIZE}_MD_{MAX_DEPTH}_T_{THRESOULD}_TT_{TOTAL_THRESOULD}_K_{K}"

EVAL_BATCH_SIZE = 64

train_datalader = generate_dataloader(load_dataset(), batch_size=STITCH_BATCH_SIZE)
test_datalader = generate_dataloader(load_dataset("test"), batch_size=EVAL_BATCH_SIZE)
data_score, _ = next(iter(train_datalader))
data_score = data_score.numpy()
print(data_score.shape)


(32, 3, 224, 224)


In [5]:
k = 0
if os.path.exists(f"../_results_chest_xray/{RESULT_NAME}.txt"):
    with open(f"../_results_chest_xray/{RESULT_NAME}.txt", "r") as f:
        k = len(f.read().split("\n"))
        print(k)


In [6]:
os.makedirs(f"../_results_chest_xray/{RESULT_NAME}", exist_ok=True)

scoreMapper = ScoreMapper(nets, data_score)
with Report(
    EVAL_BATCH_SIZE, f"../_results_chest_xray/{RESULT_NAME}.txt", "a"
) as report:
    generator = generate_networks(
        nets,
        scoreMapper,
        data_score,
        threshold=THRESOULD,
        totalThreshold=TOTAL_THRESOULD,
        maxDepth=MAX_DEPTH,
        sample=False,
        K=K,
    )
    for i, (score, net) in enumerate(generator):
        try:
            netname = f"../_results_chest_xray/{RESULT_NAME}/net{k:03}"
            report.evaluate(nets, net, netname, score, test_datalader)
            net.save(netname)
            k += 1
        except Exception as e:
            print("ERROR", e)
            pass


2025-11-23 14:53:49.460495265 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 2 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-11-23 14:53:49.467828593 [E:onnxruntime:, sequential_executor.cc:572 ExecuteKernel] Non-zero status code returned while running Reshape node. Name:'node_view' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/tensor/reshape_helper.h:45 onnxruntime::ReshapeHelper::ReshapeHelper(const onnxruntime::TensorShape&, onnxruntime::TensorShapeVector&, bool) input_shape_size == size was false. The input tensor cannot be reshaped to the requested shape. Input shape:{32,256,6,6}, requested shape:{1,9216}

Traceback (most recent call last):
  File "/tmp/ipykernel_3539700/3268713032.py", line 17, in <module>
    for i, (score, net) in enumerate(generator):
  F

RuntimeException: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running Reshape node. Name:'node_view' Status Message: /onnxruntime_src/onnxruntime/core/providers/cpu/tensor/reshape_helper.h:45 onnxruntime::ReshapeHelper::ReshapeHelper(const onnxruntime::TensorShape&, onnxruntime::TensorShapeVector&, bool) input_shape_size == size was false. The input tensor cannot be reshaped to the requested shape. Input shape:{32,256,6,6}, requested shape:{1,9216}
